In [3]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.datasets import fetch_20newsgroups

#导入数据

newsgroups = fetch_20newsgroups()
data=['sci.space','rec.autos']
newsgroups=fetch_20newsgroups(
    subset='train',
    categories=data,
    remove=('headers','footers','quotes')
)
text=newsgroups.data

#数据清洗

stop_words=set(stopwords.words('english'))
def wash(s):
    s=s.lower()
    s=re.sub('r[a-zA-Z]','',s)
    s=word_tokenize(s)
    ans=[w for w in s if len(w)>2 and w not in stop_words]
    return ans

washed=[wash(t) for t in text]
doc=[' '.join(word) for word in washed]
print(doc[0][:20])



well thank dennis us


In [4]:
#one-hot
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

v=CountVectorizer(binary=True,max_features=10)
v2=v.fit_transform(doc)
dfone=pd.DataFrame(
    v2.toarray(),
    columns=v.get_feature_names_out()
)
print("one-hot:")
print(dfone.head())

one-hot:
   also  car  could  get  know  like  one  space  think  would
0     0    0      1    0     0     0    1      0      0      1
1     0    0      0    0     1     0    0      0      0      0
2     0    1      0    0     0     0    0      0      0      0
3     1    0      1    0     1     0    0      0      0      1
4     1    0      1    0     1     1    0      0      0      1


In [5]:
#CountVectorizer
xcount=CountVectorizer(binary=True,max_features=10)
xcount2=xcount.fit_transform(doc)
dfcount=pd.DataFrame(
    xcount2.toarray(),
    columns=xcount.get_feature_names_out()
)
print("词频向量：")
print(dfcount.head())

词频向量：
   also  car  could  get  know  like  one  space  think  would
0     0    0      1    0     0     0    1      0      0      1
1     0    0      0    0     1     0    0      0      0      0
2     0    1      0    0     0     0    0      0      0      0
3     1    0      1    0     1     0    0      0      0      1
4     1    0      1    0     1     1    0      0      0      1


In [6]:
#TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tv=TfidfVectorizer(max_features=10)
X_tv=tv.fit_transform(doc)
dfx=pd.DataFrame(
    X_tv.toarray(),
    columns=tv.get_feature_names_out()
)
print('TF-IDF:')
print(dfx.head())

TF-IDF:
       also  car     could  get      like  nasa       one  space  time  \
0  0.000000  0.0  0.733644  0.0  0.000000   0.0  0.322728    0.0   0.0   
1  0.000000  0.0  0.000000  0.0  0.000000   0.0  0.000000    0.0   0.0   
2  0.000000  1.0  0.000000  0.0  0.000000   0.0  0.000000    0.0   0.0   
3  0.455843  0.0  0.465389  0.0  0.000000   0.0  0.000000    0.0   0.0   
4  0.575109  0.0  0.293576  0.0  0.260158   0.0  0.000000    0.0   0.0   

      would  
0  0.598008  
1  0.000000  
2  0.000000  
3  0.758696  
4  0.717900  


In [7]:
#word2vec
from gensim.models import Word2Vec

w2v=Word2Vec(
    sentences=doc,
    vector_size=50,
    window=5,
    min_count=2,
    workers=4
)
wv=w2v.wv

In [8]:
#转DataFrame
dfw2v=pd.DataFrame(
    wv.vectors,
    index=wv.index_to_key,
    columns=[f'dim_{i}' for i in range(wv.vector_size)]
)
print("w2v:")
print(dfw2v.head())

w2v:
      dim_0     dim_1     dim_2     dim_3     dim_4     dim_5     dim_6  \
  -0.127580 -0.135127  0.127714  0.190251 -0.390212  0.084397 -0.112951   
e -0.590269  0.486668 -0.218641  0.080293  0.318800 -0.439971  0.212406   
a -0.404082  0.237620 -0.285240 -0.120222  0.353436 -0.290534  0.086198   
s -0.085432  0.158519 -0.131368 -0.070151 -0.204291  0.106845  0.172808   
t -0.342620  0.181475  0.041069  0.127937 -0.068928  0.189285  0.201255   

      dim_7     dim_8     dim_9  ...    dim_40    dim_41    dim_42    dim_43  \
   0.209672 -0.149210  0.011282  ... -0.238925 -0.217828 -0.946323 -0.084830   
e -0.060684 -0.036735  0.020425  ... -0.386762 -0.065576  0.345308 -0.275714   
a  0.128330  0.117222 -0.113594  ...  0.092085 -0.403937  0.214184 -0.350451   
s -0.017110 -0.374980 -0.234565  ... -0.077972 -0.091921  0.330377 -0.279000   
t -0.311117 -0.468714  0.058079  ... -0.102520  0.095796 -0.396227 -0.027425   

     dim_44    dim_45    dim_46    dim_47    dim_48    dim_49  

In [9]:
#加入词频
dfw2v['count']=[wv.get_vecattr(word,'count') for word in wv.index_to_key]
print(dfw2v.head())

      dim_0     dim_1     dim_2     dim_3     dim_4     dim_5     dim_6  \
  -0.127580 -0.135127  0.127714  0.190251 -0.390212  0.084397 -0.112951   
e -0.590269  0.486668 -0.218641  0.080293  0.318800 -0.439971  0.212406   
a -0.404082  0.237620 -0.285240 -0.120222  0.353436 -0.290534  0.086198   
s -0.085432  0.158519 -0.131368 -0.070151 -0.204291  0.106845  0.172808   
t -0.342620  0.181475  0.041069  0.127937 -0.068928  0.189285  0.201255   

      dim_7     dim_8     dim_9  ...    dim_41    dim_42    dim_43    dim_44  \
   0.209672 -0.149210  0.011282  ... -0.217828 -0.946323 -0.084830  0.117701   
e -0.060684 -0.036735  0.020425  ... -0.065576  0.345308 -0.275714  0.457006   
a  0.128330  0.117222 -0.113594  ... -0.403937  0.214184 -0.350451  0.626995   
s -0.017110 -0.374980 -0.234565  ... -0.091921  0.330377 -0.279000  0.292180   
t -0.311117 -0.468714  0.058079  ...  0.095796 -0.396227 -0.027425  0.306450   

     dim_45    dim_46    dim_47    dim_48    dim_49   count  
   0.1

In [10]:
#句子向量
import numpy as np
def senv(word):
    v=[wv[w] for w in word if w in wv]
    return np.mean(v,axis=0) if v else np.zeros(wv.vector_size)

sen=np.array([senv(word) for word in doc[:10]])

dfsen=pd.DataFrame(
    sen,
    columns=[f'dim_[i]' for i in range(wv.vector_size)]
)
print('句子向量：')
print(dfsen.head)

句子向量：
<bound method NDFrame.head of     dim_[i]   dim_[i]   dim_[i]   dim_[i]   dim_[i]   dim_[i]   dim_[i]  \
0 -0.265332  0.154998 -0.082510  0.024685 -0.058945 -0.041234  0.066745   
1 -0.230617 -0.000174 -0.028788 -0.030597 -0.108473 -0.020532  0.017833   
2 -0.293350  0.156158 -0.075214  0.004229 -0.085947 -0.049992  0.039072   
3 -0.286627  0.137229 -0.083480  0.012768 -0.056183 -0.051812  0.058222   
4 -0.270725  0.123651 -0.067892 -0.015614 -0.070798 -0.052823  0.023876   
5 -0.290070  0.121059 -0.048212  0.044259  0.007666 -0.079723  0.068517   
6 -0.297904  0.172599 -0.110920  0.019175 -0.058264 -0.051718  0.061374   
7 -0.279840  0.143727 -0.102539 -0.009865 -0.097085 -0.038556  0.036500   
8 -0.290587  0.141150 -0.103912 -0.006116 -0.081378 -0.044983  0.068072   
9 -0.296671  0.268033 -0.091949 -0.075795 -0.011935 -0.111573 -0.026855   

    dim_[i]   dim_[i]   dim_[i]  ...   dim_[i]   dim_[i]   dim_[i]   dim_[i]  \
0 -0.056567 -0.190147 -0.094633  ... -0.141830 -0.023840 -

In [11]:
w2v.save("word2vec_20news.model")

wv.save("word_vectors.kv")

w2v.wv.save_word2vec_format("vectors.bin",binary=True)